In [ ]:
import os
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Environment ready")
print("TensorFlow version:", tf.__version__)

In [ ]:
DATASET_PATH = r"train_with_images.csv" 
df = pd.read_csv(DATASET_PATH)

print("Loaded dataframe shape:", df.shape)
print("Columns:", df.columns.tolist())


print("Dataset loaded & verified")


## Target engineering

In [ ]:
assert (df["price"] >= 0).all(), "Negative prices found"

# Log-transform target
df["log_price"] = np.log1p(df["price"])

df = df.dropna(subset=["image_path", "log_price"]).reset_index(drop=True)

assert np.isfinite(df["log_price"]).all()
assert df["image_path"].notna().all()

print("Cell 3 executed:")
print("Final dataframe shape:", df.shape)
print(df[["price", "log_price"]].head())


## Reproducibility

In [ ]:
import os
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)


## Define tabular feature columns

In [ ]:
TABULAR_COLUMNS = [
    "bedrooms", "bathrooms", "sqft_living", "sqft_lot",
    "floors", "waterfront", "view", "condition", "grade",
    "sqft_above", "sqft_basement",
    "yr_built",
    "lat", "long",
    "sqft_living15", "sqft_lot15"
]

print("Cell 4 executed: Tabular columns verified")
print("Number of tabular features:", len(TABULAR_COLUMNS))


## Train / validation split

In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_val = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED
)

print("Train shape:", df_train.shape)
print("Validation shape:", df_val.shape)


## Scale tabular features

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_tab = scaler.fit_transform(df_train[TABULAR_COLUMNS])
X_val_tab   = scaler.transform(df_val[TABULAR_COLUMNS])

# Targets (log-price)
y_train = df_train["log_price"].values.astype("float32")
y_val   = df_val["log_price"].values.astype("float32")

assert np.isfinite(X_train_tab).all()
assert np.isfinite(X_val_tab).all()
assert np.isfinite(y_train).all()
assert np.isfinite(y_val).all()

print("Cell 6 executed:")
print("X_train_tab shape:", X_train_tab.shape)
print("X_val_tab shape:", X_val_tab.shape)
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)


## Build tabular-only baseline model

In [ ]:
from tensorflow.keras import layers, models

tabular_input = tf.keras.Input(
    shape=(X_train_tab.shape[1],),
    name="tabular_input",
    dtype="float32"
)

x = layers.Dense(128, activation="relu")(tabular_input)
x = layers.Dense(64, activation="relu")(x)
x = layers.Dense(32, activation="relu")(x)

output = layers.Dense(1, name="log_price")(x)

tabular_model = models.Model(
    inputs=tabular_input,
    outputs=output
)

tabular_model.summary()

## Compile baseline model

In [ ]:
from tensorflow.keras.optimizers import Adam

tabular_model.compile(
    optimizer=Adam(
        learning_rate=1e-4,   # conservative for stability
        clipnorm=1.0          # prevents gradient explosion
    ),
    loss=tf.keras.losses.Huber(delta=1.0)  
)

print("Cell 8 executed: Baseline model compiled safely")

## Train tabular-only baseline model

In [ ]:
history_tabular = tabular_model.fit(
    X_train_tab,
    y_train,
    validation_data=(X_val_tab, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=6,
            restore_best_weights=True
        )
    ],
    verbose=1
)


## Evaluate tabular-only baseline (LOG SPACE)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

y_val_pred = tabular_model.predict(X_val_tab).reshape(-1)

# Metrics
rmse_log = np.sqrt(mean_squared_error(y_val, y_val_pred))
r2_log   = r2_score(y_val, y_val_pred)

print("TABULAR-ONLY RMSE (log space):", rmse_log)
print("TABULAR-ONLY R² (log space):", r2_log)


## HYBRID MODEL SETUP (Tabular + Images)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

IMG_SIZE = (224, 224)
IMG_SHAPE = (224, 224, 3)

BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

print("Hybrid model setup ready")
print("IMG_SHAPE:", IMG_SHAPE)
print("BATCH_SIZE:", BATCH_SIZE)


## Load dataset for Hybrid model

In [ ]:
import pandas as pd
import os

DATASET_PATH = r"train_with_images.csv"

df = pd.read_csv(DATASET_PATH)

print("Dataset loaded")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

assert "image_path" in df.columns, "image_path column missing"
assert "price" in df.columns, "price column missing"

missing_imgs = df[~df["image_path"].apply(os.path.exists)]
print("Missing images:", len(missing_imgs))

assert len(missing_imgs) == 0, "Some image files are missing!"


## Define tabular features & target

In [ ]:
import numpy as np

TABULAR_COLUMNS = [
    "bedrooms", "bathrooms", "sqft_living", "sqft_lot",
    "floors", "waterfront", "view", "condition", "grade",
    "sqft_above", "sqft_basement",
    "yr_built",
    "lat", "long",
    "sqft_living15", "sqft_lot15"
]

missing_cols = set(TABULAR_COLUMNS) - set(df.columns)
assert len(missing_cols) == 0, f"Missing columns: {missing_cols}"

df["log_price"] = np.log1p(df["price"])

print("Target created: log_price")

print("PRICE range:", df["price"].min(), "→", df["price"].max())
print("LOG_PRICE range:", df["log_price"].min(), "→", df["log_price"].max())


## Train / Validation split

In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_val = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED
)

print("Train shape:", df_train.shape)
print("Validation shape:", df_val.shape)


In [ ]:
# ======================================================
# Scale tabular features (CRITICAL)
# ======================================================

from sklearn.preprocessing import StandardScaler
import numpy as np

scaler = StandardScaler()

# Fit ONLY on training data
X_train_tab = scaler.fit_transform(df_train[TABULAR_COLUMNS])
X_val_tab   = scaler.transform(df_val[TABULAR_COLUMNS])

# Targets (log space)
y_train = df_train["log_price"].values.astype("float32")
y_val   = df_val["log_price"].values.astype("float32")

# Safety checks
assert np.isfinite(X_train_tab).all()
assert np.isfinite(X_val_tab).all()
assert np.isfinite(y_train).all()
assert np.isfinite(y_val).all()

print("Tabular scaling complete")
print("X_train_tab shape:", X_train_tab.shape)
print("X_val_tab shape:", X_val_tab.shape)

## Image loading function

In [ ]:
IMG_SIZE = (224, 224)

def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image




## Build hybrid tf.data datasets (image + tabular)

In [ ]:
def make_hybrid_dataset(image_paths, tabular_data, targets, shuffle=True):
    tabular_data = tabular_data.astype("float32")
    targets = targets.astype("float32")

    ds = tf.data.Dataset.from_tensor_slices(
        (
            {
                "image_input": image_paths,
                "tabular_input": tabular_data,
            },
            targets
        )
    )

    def _map_fn(x, y):
        x["image_input"] = load_image(x["image_input"])
        return x, y

    ds = ds.map(_map_fn, num_parallel_calls=AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(buffer_size=2048, seed=SEED)

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)

    return ds


train_ds = make_hybrid_dataset(
    df_train["image_path"].values,
    X_train_tab,
    y_train,
    shuffle=True
)

val_ds = make_hybrid_dataset(
    df_val["image_path"].values,
    X_val_tab,
    y_val,
    shuffle=False
)

print("Hybrid datasets created")


## Build HYBRID model (Image + Tabular)

In [ ]:
image_input = keras.Input(shape=IMG_SHAPE, name="image_input")

base_cnn = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_tensor=image_input
)

for layer in base_cnn.layers[:-40]:
    layer.trainable = False
for layer in base_cnn.layers[-40:]:
    layer.trainable = True

x_img = base_cnn.output
x_img = layers.GlobalAveragePooling2D()(x_img)

x_img = layers.Dense(256, activation="relu")(x_img)
x_img = layers.Dense(128, activation="relu")(x_img)

tabular_input = keras.Input(
    shape=(X_train_tab.shape[1],),
    name="tabular_input"
)

x_tab = layers.Dense(128, activation="relu")(tabular_input)
x_tab = layers.Dense(64, activation="relu")(x_tab)

combined = layers.Concatenate()([x_img, x_tab])

combined = layers.Dense(128, activation="relu")(combined)
combined = layers.Dense(64, activation="relu")(combined)

output = layers.Dense(1, name="log_price")(combined)

hybrid_model = keras.Model(
    inputs={"image_input": image_input, "tabular_input": tabular_input},
    outputs=output
)

hybrid_model.summary()


## Compile HYBRID model

In [ ]:
hybrid_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4,
        clipnorm=1.0
    ),
    loss=tf.keras.losses.Huber(delta=1.0),
    metrics=[]
)

print("Hybrid model compiled successfully")


## HYBRID TRAINING: STAGE-1 ➜ STAGE-2

In [ ]:
from tensorflow import keras
import os
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)
for layer in hybrid_model.layers:
    if "efficientnet" in layer.name.lower():
        layer.trainable = False

hybrid_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-4,
        clipnorm=1.0
    ),
    loss=keras.losses.Huber(delta=1.0)
)

stage1_ckpt = "best_hybrid_model_stage1.h5"

history_stage1 = hybrid_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[
        early_stop,
        keras.callbacks.ModelCheckpoint(
            stage1_ckpt,
            monitor="val_loss",
            save_best_only=True,
            verbose=1
        )
    ],
    verbose=1
)

print("Stage-1 completed and saved.")

print("STAGE 2: Fine-tuning CNN")

hybrid_model = keras.models.load_model(stage1_ckpt)

for layer in hybrid_model.layers:
    if "efficientnet" in layer.name.lower():
        layer.trainable = True

cnn_layers = [
    layer for layer in hybrid_model.layers
    if "efficientnet" in layer.name.lower()
]

for layer in cnn_layers[:-20]:
    layer.trainable = False

hybrid_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-5,
        clipnorm=1.0
    ),
    loss=keras.losses.Huber(delta=1.0)
)

final_ckpt = "best_hybrid_model_final.h5"

history_stage2 = hybrid_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    callbacks=[
        early_stop,
        keras.callbacks.ModelCheckpoint(
            final_ckpt,
            monitor="val_loss",
            save_best_only=True,
            verbose=1
        )
    ],
    verbose=1
)

print("Stage-2 completed and final model saved.")


## Evaluate HYBRID model (LOG SPACE R²)

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

y_true = []
y_pred = []

for x, y in val_ds:
    preds = hybrid_model.predict(x, verbose=0).squeeze()

    y_true.append(y.numpy())
    y_pred.append(preds)

y_true = np.concatenate(y_true)
y_pred = np.concatenate(y_pred)

mask = np.isfinite(y_true) & np.isfinite(y_pred)
y_true = y_true[mask]
y_pred = y_pred[mask]

assert len(y_true) > 0, "No valid predictions"
assert np.var(y_true) > 0, "Zero variance in y_true"

rmse_log = np.sqrt(mean_squared_error(y_true, y_pred))
r2_log = r2_score(y_true, y_pred)

print("HYBRID RMSE (log space):", rmse_log)
print("HYBRID R² (log space):", r2_log)



In [ ]:
import pandas as pd
import os

CSV_PATH = r"train_with_images.csv"
assert os.path.exists(CSV_PATH), f" CSV not found at {CSV_PATH}"

df = pd.read_csv(CSV_PATH)

print(" CSV loaded successfully")
print("Rows:", len(df))
print("Columns:")
print(df.columns.tolist())


In [ ]:
import os

IMAGE_ROOT = os.path.dirname(CSV_PATH)

print("IMAGE_ROOT set to:")
print(IMAGE_ROOT)

print(
    "satellite_images folder exists?",
    os.path.exists(os.path.join(IMAGE_ROOT, "satellite_images"))
)


In [ ]:
import tensorflow as tf
import numpy as np
import random

IMG_SIZE = 224
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("Global config set")
print("IMG_SIZE:", IMG_SIZE)
print("BATCH_SIZE:", BATCH_SIZE)
print("SEED:", SEED)


In [ ]:


import tensorflow as tf

def load_image(path):
    img_bytes = tf.io.read_file(path)
    img = tf.image.decode_png(img_bytes, channels=3)

    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32) / 255.0

    return img



In [ ]:
sample_path = df.iloc[0]["image_path"]

test_img = load_image(sample_path)

print("Image shape:", test_img.shape)
print(
    "Min / Max:",
    tf.reduce_min(test_img).numpy(),
    tf.reduce_max(test_img).numpy()
)


In [ ]:
import numpy as np

df["log_price"] = np.log1p(df["price"])

print("log_price created")
print("price min / max:", df["price"].min(), df["price"].max())
print("log_price min / max:", df["log_price"].min(), df["log_price"].max())
